In [1]:
import torch
from collections import defaultdict

In [2]:
def load_data(file_pattern, num_files):
    data_by_coords = defaultdict(list)

    for i in range(1, num_files + 1):
        filename = file_pattern.format(i)
        data = torch.load(filename)
        
        for entry in data:
            coords = tuple(entry['coordinates'])
            answer = entry['answer'].squeeze(0)  # Reshape [1, 8, 6] to [8, 6]
            data_by_coords[coords].append(answer)

    return data_by_coords


In [3]:
data_by_coords = load_data("dataset/3p6_time_{}.torch", 10)

In [4]:
len(data_by_coords[-117, -76, -25])

10

In [5]:
len(data_by_coords)

33598

In [6]:
len(data_by_coords[-117, -76, -25])

10

In [7]:
data_by_coords[-117, -76, -25][0]

tensor([[ 0.0208, -0.0564,  0.0217,  0.0407, -0.0383,  0.0788],
        [-0.0520,  0.1376, -0.3669, -0.0146, -0.2297, -0.0339],
        [ 0.1281,  0.0225, -0.1117, -0.1245, -0.0141,  0.0845],
        [-0.0250, -0.0405,  0.1901, -0.0901,  0.1176,  0.0058],
        [ 0.0156,  0.0457,  0.2644, -0.7112, -0.0264, -0.0290],
        [ 0.0317,  0.0128,  0.1344,  0.0088, -0.0224,  0.0215],
        [ 0.1371,  0.1382,  0.0763,  0.1292,  0.0388,  0.0030],
        [-0.1457,  0.1424, -0.0142, -0.0054,  0.1672,  0.0336]],
       grad_fn=<SqueezeBackward1>)

In [8]:
# 2. Organize data into source-target pairs
def create_source_target(data_by_coords, source_length=4, target_length=1):
    pairs = [] 
    for coords, tensor_list in data_by_coords.items():
        for i in range(0, len(tensor_list) - source_length - target_length + 1, source_length + target_length):
            source = tensor_list[i: i + source_length]
            target = tensor_list[i + source_length: i + source_length + target_length]
            pairs.append((source, target))
    
    return pairs

In [9]:
data_pairs = create_source_target(data_by_coords)

In [10]:
type(data_pairs)

list

In [11]:
len(data_pairs)

67196

In [44]:
for i in range(0, 10 - 4 - 1 + 1, 4 + 1):
    print(i)

0
5


In [42]:
data_pairs[0]

([tensor([[ 0.0208, -0.0564,  0.0217,  0.0407, -0.0383,  0.0788],
          [-0.0520,  0.1376, -0.3669, -0.0146, -0.2297, -0.0339],
          [ 0.1281,  0.0225, -0.1117, -0.1245, -0.0141,  0.0845],
          [-0.0250, -0.0405,  0.1901, -0.0901,  0.1176,  0.0058],
          [ 0.0156,  0.0457,  0.2644, -0.7112, -0.0264, -0.0290],
          [ 0.0317,  0.0128,  0.1344,  0.0088, -0.0224,  0.0215],
          [ 0.1371,  0.1382,  0.0763,  0.1292,  0.0388,  0.0030],
          [-0.1457,  0.1424, -0.0142, -0.0054,  0.1672,  0.0336]],
         grad_fn=<SqueezeBackward1>),
  tensor([[ 0.0208, -0.0564,  0.0217,  0.0407, -0.0383,  0.0788],
          [-0.0520,  0.1376, -0.3669, -0.0146, -0.2297, -0.0339],
          [ 0.1281,  0.0225, -0.1117, -0.1245, -0.0141,  0.0845],
          [-0.0250, -0.0405,  0.1901, -0.0901,  0.1176,  0.0058],
          [ 0.0156,  0.0457,  0.2644, -0.7112, -0.0264, -0.0290],
          [ 0.0317,  0.0128,  0.1344,  0.0088, -0.0224,  0.0215],
          [ 0.1371,  0.1382,  0.0763,

In [15]:
from torch.utils.data import DataLoader, Dataset

class SequenceDataset(Dataset):
    def __init__(self, files, source_len = 4, target_len = 1):
        self.files = files
        self.source_len = source_len
        self.target_len = target_len

    def __len__(self):
        return len(self.files) - (self.source_len + self.target_len) + 1

    def __getitem__(self, idx):
        sources = []
        targets = []

        # Collecting the source files
        for i in range(self.source_len):
            data = torch.load(self.files[idx + i])
            sources.append(data['answer'].squeeze().view(-1))  # Flattening [8,6] to [48]

        # Collecting the target files
        for j in range(self.target_len):
            data = torch.load(self.files[idx + self.source_len + j])
            targets.append(data['answer'].squeeze().view(-1))  # Flattening [8,6] to [48]

        # Convert lists to tensors
        source = torch.stack(sources)
        target = torch.stack(targets)

        return source, target

In [16]:
dataset = SequenceDataset(data_by_coords)
dataloader = DataLoader(dataset, batch_size=40, shuffle=True)

In [17]:
import torch.nn as nn

class Seq2SeqTransformer(nn.Module):
    def __init__(self, input_dim, output_dim, d_model, nhead, num_encoder_layers, num_decoder_layers, dropout):
        super(Seq2SeqTransformer, self).__init__()

        self.transformer = nn.Transformer(d_model, nhead, num_encoder_layers, num_decoder_layers, dropout=dropout)
        self.source_embedding = nn.Linear(input_dim, d_model)
        self.target_embedding = nn.Linear(output_dim, d_model)
        self.output_layer = nn.Linear(d_model, output_dim)

    def forward(self, src, tgt):
        src = self.source_embedding(src)
        tgt = self.target_embedding(tgt)
        output = self.transformer(src, tgt)
        return self.output_layer(output)




In [18]:
# Hyperparameters
INPUT_DIM = 6   # Source tensor dimension
OUTPUT_DIM = 6  # Target tensor dimension
D_MODEL = 512
NHEAD = 8
NUM_ENCODER_LAYERS = 3
NUM_DECODER_LAYERS = 3
DROPOUT = 0.1

model = Seq2SeqTransformer(INPUT_DIM, OUTPUT_DIM, D_MODEL, NHEAD, NUM_ENCODER_LAYERS, NUM_DECODER_LAYERS, DROPOUT)

In [20]:
import torch.optim as optim

# Define the loss function and the optimizer
loss_function = nn.MSELoss()  # Mean squared error loss, suitable for regression tasks
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training hyperparameters
EPOCHS = 100
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "mps")
model.to(DEVICE)

for epoch in range(EPOCHS):
    total_loss = 0.0

    for batch in dataloader:
        src_list, tgt_list = batch
        print(len(src_list))
        print(len(tgt_list))
        
        src = torch.stack(src_list).to(DEVICE)
        tgt = torch.stack(tgt_list).to(DEVICE)

        print(src.shape)
        print(tgt.shape)

        optimizer.zero_grad()

        # Since we don't have a separate tgt_input for teacher forcing, 
        # we can use zeros of the same shape as tgt as a placeholder.
        tgt_input = torch.zeros_like(tgt).to(DEVICE)

        # Forward pass
        output = model(src, tgt_input)

        # Compute the loss
        loss = loss_function(output, tgt)

        # Backward pass
        loss.backward()

        # Optimization step
        optimizer.step()

        total_loss += loss.item()

    # Print average loss for the epoch
    print(f"Epoch {epoch + 1}/{EPOCHS} - Loss: {total_loss / len(dataloader)}")


AttributeError: 'list' object has no attribute 'seek'. You can only torch.load from a file that is seekable. Please pre-load the data into a buffer like io.BytesIO and try to load from it instead.

In [21]:
import torch
from collections import defaultdict

def load_data(file_pattern, num_files):
    data_by_coords = defaultdict(list)

    for i in range(1, num_files + 1):
        filename = file_pattern.format(i)
        data = torch.load(filename)

        for entry in data:
            coords = tuple(entry['coordinates'])
            answer = entry['answer'].squeeze(0)  # Reshape [1, 8, 6] to [8, 6]
            data_by_coords[coords].append(answer)

    return data_by_coords


In [24]:
data = load_data("dataset/3p6_time_{}.torch", 10)


In [25]:
from torch.utils.data import Dataset, DataLoader
import os

class SequenceDataset(Dataset):
    def __init__(self, data, source_len=4, target_len=1):
        self.data = data
        self.data_list = list(data.values())
        self.source_len = source_len
        self.target_len = target_len

    def __len__(self):
        return len(self.data_list) * (len(self.data_list[0]) - self.source_len - self.target_len + 1)

    def __getitem__(self, idx):
        coord_idx = idx // (len(self.data_list[0]) - self.source_len - self.target_len + 1)
        time_idx = idx % (len(self.data_list[0]) - self.source_len - self.target_len + 1)

        source = torch.cat(self.data_list[coord_idx][time_idx:time_idx+self.source_len], dim=1)
        target = torch.cat(self.data_list[coord_idx][time_idx+self.source_len:time_idx+self.source_len+self.target_len], dim=1)

        return source, target

In [26]:
dataset = SequenceDataset(data)
data_loader = DataLoader(dataset, batch_size=32, shuffle=True)

In [27]:
import torch.nn as nn

class TransformerModel(nn.Module):
    def __init__(self, input_dim, model_dim, num_heads, num_layers, output_dim):
        super(TransformerModel, self).__init__()

        self.encoder = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(model_dim, num_heads),
            num_layers
        )
        self.decoder = nn.TransformerDecoder(
            nn.TransformerDecoderLayer(model_dim, num_heads),
            num_layers
        )
        self.fc = nn.Linear(input_dim, output_dim)

    def forward(self, src, tgt):
        src = self.fc(src)
        tgt = self.fc(tgt)
        memory = self.encoder(src)
        out = self.decoder(tgt, memory)
        return out


In [28]:
device = torch.device("cuda" if torch.cuda.is_available() else "mps")
# Initialize model, criterion, and optimizer
model = TransformerModel(48, 256, 4, 4, 48).to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
i = 0
# Training loop
for epoch in range(10):
    for source, target in data_loader:
        i+=1
        source, target = source.to(device), target.to(device)
        if i == 1:
            print(source.shape)
            print(target.shape)


        optimizer.zero_grad()
        outputs = model(source, target)
        loss = criterion(outputs, target)
        loss.backward()
        optimizer.step()

        print(f"Epoch [{epoch + 1}/10], Loss: {loss.item():.4f}")

In [8]:
# Specify the path to the saved new .torch file
# new_file_path = "dataset/3p6_time_1.torch"
new_file_path = "/mnt/d/sources/cgan/playground/dataset/3p6_time_1.torch"

In [4]:
# Load the new data from file
loaded_new_data = torch.load(new_file_path)

In [5]:
# Check if there are at least 5 items in the loaded data
if len(loaded_new_data) < 5:
    print("There are less than 5 items in the loaded data.")
else:
    # Print out the first 5 items
    for i in range(1):
        print("Item", i + 1)
        print("Coordinates:", loaded_new_data[i]['coordinates'])
        print("Velocity:", loaded_new_data[i]['velocity'])
        print("Answer:", loaded_new_data[i]['answer'])
        print('---' * 20)


Item 1
Coordinates: [-117, -76, -25]
Velocity: tensor([[[0.5322, 0.5322, 0.5322, 0.5347, 0.5347, 0.5322, 0.5322, 0.5322,
          0.5347, 0.5347, 0.5322, 0.5322, 0.5347, 0.5347, 0.5366, 0.5322,
          0.5322, 0.5347, 0.5347, 0.5366, 0.5322, 0.5322, 0.5347, 0.5347,
          0.5366, 0.5322, 0.5322, 0.5322, 0.5347, 0.5347, 0.5322, 0.5322,
          0.5322, 0.5347, 0.5347, 0.5322, 0.5322, 0.5322, 0.5347, 0.5347,
          0.5322, 0.5322, 0.5347, 0.5347, 0.5347, 0.5322, 0.5322, 0.5347,
          0.5347, 0.5366, 0.5322, 0.5322, 0.5322, 0.5347, 0.5347, 0.5322,
          0.5322, 0.5322, 0.5347, 0.5347, 0.5322, 0.5322, 0.5322, 0.5347,
          0.5347, 0.5322, 0.5322, 0.5322, 0.5347, 0.5347, 0.5322, 0.5322,
          0.5322, 0.5347, 0.5347, 0.5322, 0.5322, 0.5322, 0.5322, 0.5347,
          0.5322, 0.5322, 0.5322, 0.5322, 0.5347, 0.5322, 0.5322, 0.5322,
          0.5322, 0.5347, 0.5322, 0.5322, 0.5322, 0.5322, 0.5347, 0.5322,
          0.5322, 0.5322, 0.5322, 0.5347, 0.5347, 0.5347, 0.5347,

In [6]:
len(loaded_new_data)

33598

In [7]:
loaded_new_data.dtype

AttributeError: 'list' object has no attribute 'dtype'

In [8]:
type(loaded_new_data)

list

In [9]:
loaded_new_data[0]

{'coordinates': [-117, -76, -25],
 'velocity': tensor([[[0.5322, 0.5322, 0.5322, 0.5347, 0.5347, 0.5322, 0.5322, 0.5322,
           0.5347, 0.5347, 0.5322, 0.5322, 0.5347, 0.5347, 0.5366, 0.5322,
           0.5322, 0.5347, 0.5347, 0.5366, 0.5322, 0.5322, 0.5347, 0.5347,
           0.5366, 0.5322, 0.5322, 0.5322, 0.5347, 0.5347, 0.5322, 0.5322,
           0.5322, 0.5347, 0.5347, 0.5322, 0.5322, 0.5322, 0.5347, 0.5347,
           0.5322, 0.5322, 0.5347, 0.5347, 0.5347, 0.5322, 0.5322, 0.5347,
           0.5347, 0.5366, 0.5322, 0.5322, 0.5322, 0.5347, 0.5347, 0.5322,
           0.5322, 0.5322, 0.5347, 0.5347, 0.5322, 0.5322, 0.5322, 0.5347,
           0.5347, 0.5322, 0.5322, 0.5322, 0.5347, 0.5347, 0.5322, 0.5322,
           0.5322, 0.5347, 0.5347, 0.5322, 0.5322, 0.5322, 0.5322, 0.5347,
           0.5322, 0.5322, 0.5322, 0.5322, 0.5347, 0.5322, 0.5322, 0.5322,
           0.5322, 0.5347, 0.5322, 0.5322, 0.5322, 0.5322, 0.5347, 0.5322,
           0.5322, 0.5322, 0.5322, 0.5347, 0.5347, 0.5